# 22 — Condition-specific malignant pseudobulk differential expression

This notebook is the repository-numbered equivalent of the historical executed notebook:

`21_condition_specific_malignant_pseudobulk_DE_HVG8000_r0_6_v3_fixed_duplicate_state_column.ipynb`

A direct cell-by-cell audit showed that the repository Notebook 22 and the historical executed notebook contain the same scientific/statistical code; the repository version only added repository-root path handling and a wrapper title.

## Historical scientific semantics preserved

The analysis retains the historical:

- five malignant/cycling cluster-defined states;
- overall malignant/cycling sample-level pseudobulk analysis;
- sample × malignant-state pseudobulk analysis;
- minimum 30 cells per pseudobulk;
- CPM expression filter: CPM >= 1 in at least 3 pseudobulks;
- global Kruskal-Wallis tests;
- pairwise Welch t tests;
- pairwise Mann-Whitney U tests;
- Benjamini-Hochberg correction;
- absolute log2FC >= 0.5 significance export;
- overall top 100 up/down genes per contrast;
- within-state top 50 up/down genes per contrast;
- seven historical program-marker panels and z-scored program scores;
- three historical figures.

## Canonical upstream policy

This notebook requires two outputs from the canonical pipeline:

1. Notebook 20 final malignant-state annotation H5AD:

   `GBM_core_4datasets_final_annotation_malignant_states.h5ad`

2. Notebook 21 malignant-state pseudobulk H5AD:

   `GBM_core_4datasets_malignant_state_pseudobulk_logCPM.h5ad`

Notebook 21 already reproduces the exact historical `sample × malignant_state` raw-count pseudobulks. Notebook 22 therefore reuses those raw pseudobulk counts instead of loading and subsetting the 351,427-cell raw-count matrix again.

The overall malignant/cycling sample pseudobulks are reconstructed by exact integer summation across the five state pseudobulks belonging to each sample. This is algebraically identical to the historical direct cell-to-sample aggregation because the five states partition the selected malignant/cycling cells.

## Memory-safety

The full Notebook-20 H5AD is opened only for `/obs` and `/var` metadata. Its `counts` layer is never materialized.

The Notebook-21 pseudobulk object contains only 375 × 15,176 rows/genes and is safe to read. Only its raw pseudobulk counts are used; Notebook 22 recomputes the historical base-2 `log2(CPM + 1)` matrices itself.

This preserves the historical statistical semantics while eliminating the high-RAM full-cell expression copy.

In [1]:
# =========================
# 0. Imports and paths
# =========================
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats
import anndata as ad
import h5py
import matplotlib.pyplot as plt

try:
    from anndata.io import read_elem
except Exception:
    from anndata._io.specs import read_elem


RANDOM_STATE = 42
np.random.seed(
    RANDOM_STATE
)


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
    / "condition_malignant_pseudobulk_DE_HVG8000_r0_6"
)

for directory in [
    METADATA_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


INPUT_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"
)

UPSTREAM_PSEUDOBULK_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_malignant_state_pseudobulk_logCPM.h5ad"
)

OUT_PREFIX = (
    "condition_malignant_pseudobulk_DE_HVG8000_r0_6"
)


print(
    "Historical notebook mapping:",
    "21_condition_specific_malignant_pseudobulk_DE_HVG8000_r0_6_"
    "v3_fixed_duplicate_state_column.ipynb -> 22_condition_pseudobulk.ipynb",
)

print(
    "INPUT_H5AD:",
    INPUT_H5AD,
)

print(
    "UPSTREAM_PSEUDOBULK_H5AD:",
    UPSTREAM_PSEUDOBULK_H5AD,
)

print(
    "METADATA_DIR:",
    METADATA_DIR,
)

print(
    "FIGURES_DIR:",
    FIGURES_DIR,
)


if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        "Notebook 22 requires the canonical Notebook-20 malignant-state H5AD:\n"
        f"{INPUT_H5AD}"
    )


if not UPSTREAM_PSEUDOBULK_H5AD.exists():
    raise FileNotFoundError(
        "Notebook 22 requires the canonical Notebook-21 pseudobulk H5AD:\n"
        f"{UPSTREAM_PSEUDOBULK_H5AD}"
    )


Historical notebook mapping: 21_condition_specific_malignant_pseudobulk_DE_HVG8000_r0_6_v3_fixed_duplicate_state_column.ipynb -> 22_condition_pseudobulk.ipynb
INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad
UPSTREAM_PSEUDOBULK_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_malignant_state_pseudobulk_logCPM.h5ad
METADATA_DIR: G:\Repository\glioma-cnv-single-cell-atlas\metadata
FIGURES_DIR: G:\Repository\glioma-cnv-single-cell-atlas\figures\condition_malignant_pseudobulk_DE_HVG8000_r0_6


In [2]:
# =========================
# 1. Historical helper functions
# =========================
def write_tsv_replace(
    df,
    path,
    index=False,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        path.name
        + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=index,
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )


def write_gzip_tsv_replace(
    df,
    path,
    index=True,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        path.name
        + ".tmp.gz"
    )

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=index,
        compression={
            "method": "gzip",
            "mtime": 0,
        },
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )


def choose_first_existing_column(
    df,
    candidates,
    required=True,
    label="column",
):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    if required:
        raise KeyError(
            f"Could not find {label}. Tried: {candidates}"
        )

    return None


def benjamini_hochberg(
    pvalues,
):
    p = np.asarray(
        pvalues,
        dtype=float,
    )

    q = np.full(
        p.shape,
        np.nan,
        dtype=float,
    )

    valid = np.isfinite(
        p
    )

    if valid.sum() == 0:
        return q

    pv = p[
        valid
    ]

    n = len(
        pv
    )

    order = np.argsort(
        pv
    )

    ranked = pv[
        order
    ]

    adj = (
        ranked
        * n
        / (
            np.arange(
                n
            )
            + 1
        )
    )

    adj = np.minimum.accumulate(
        adj[
            ::-1
        ]
    )[
        ::-1
    ]

    adj = np.minimum(
        adj,
        1.0,
    )

    q_valid = np.empty(
        n,
        dtype=float,
    )

    q_valid[
        order
    ] = adj

    q[
        valid
    ] = q_valid

    return q


def safe_ttest_ind(
    x,
    y,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    y = y[
        np.isfinite(
            y
        )
    ]

    if (
        len(
            x
        ) < 2
        or len(
            y
        ) < 2
    ):
        return np.nan

    if (
        np.nanstd(
            x
        )
        == 0
        and np.nanstd(
            y
        )
        == 0
    ):
        return (
            1.0
            if np.nanmean(
                x
            )
            == np.nanmean(
                y
            )
            else np.nan
        )

    try:
        return stats.ttest_ind(
            x,
            y,
            equal_var=False,
            nan_policy="omit",
        ).pvalue

    except Exception:
        return np.nan


def safe_mannwhitney(
    x,
    y,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    y = y[
        np.isfinite(
            y
        )
    ]

    if (
        len(
            x
        ) < 2
        or len(
            y
        ) < 2
    ):
        return np.nan

    try:
        return stats.mannwhitneyu(
            x,
            y,
            alternative="two-sided",
        ).pvalue

    except Exception:
        return np.nan


def safe_kruskal(
    groups,
):
    clean = []

    for group in groups:
        group = np.asarray(
            group,
            dtype=float,
        )

        group = group[
            np.isfinite(
                group
            )
        ]

        if len(
            group
        ) > 0:
            clean.append(
                group
            )

    if len(
        clean
    ) < 2:
        return np.nan

    try:
        return stats.kruskal(
            *clean
        ).pvalue

    except Exception:
        return np.nan


def counts_to_logcpm(
    counts_df,
):
    counts = counts_df.to_numpy(
        dtype=float
    )

    lib = counts.sum(
        axis=1
    )

    lib[
        lib
        == 0
    ] = np.nan

    cpm = (
        counts
        / lib[
            :,
            None
        ]
        * 1e6
    )

    return pd.DataFrame(
        np.log2(
            cpm
            + 1.0
        ),
        index=counts_df.index,
        columns=counts_df.columns,
    )


def filter_genes_by_cpm(
    counts_df,
    min_cpm=1.0,
    min_samples=3,
):
    counts = counts_df.to_numpy(
        dtype=float
    )

    lib = counts.sum(
        axis=1
    )

    lib[
        lib
        == 0
    ] = np.nan

    cpm = (
        counts
        / lib[
            :,
            None
        ]
        * 1e6
    )

    keep = (
        np.nansum(
            cpm
            >= min_cpm,
            axis=0,
        )
        >= min_samples
    )

    return pd.Series(
        keep,
        index=counts_df.columns,
    )


def run_global_kruskal(
    logcpm_df,
    meta_df,
    condition_col,
    genes,
):
    conditions = sorted(
        meta_df[
            condition_col
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    rows = []

    for gene in genes:
        groups = [
            logcpm_df.loc[
                meta_df[
                    condition_col
                ]
                .astype(str)
                == condition,
                gene,
            ].values
            for condition
            in conditions
        ]

        p_value = safe_kruskal(
            groups
        )

        means = {
            f"mean_logcpm_{condition}": (
                float(
                    np.nanmean(
                        group
                    )
                )
                if len(
                    group
                )
                else np.nan
            )
            for condition, group
            in zip(
                conditions,
                groups,
            )
        }

        rows.append(
            {
                "gene": gene,
                "test": "Kruskal-Wallis",
                "n_conditions": len(
                    conditions
                ),
                "pvalue": p_value,
                **means,
            }
        )

    df = pd.DataFrame(
        rows
    )

    df[
        "padj"
    ] = benjamini_hochberg(
        df[
            "pvalue"
        ].values
    )

    return df.sort_values(
        [
            "padj",
            "pvalue",
        ],
        na_position="last",
    )


def run_pairwise_de(
    logcpm_df,
    meta_df,
    condition_col,
    genes,
    min_samples_per_group=3,
):
    conditions = sorted(
        meta_df[
            condition_col
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    contrasts = []

    for i in range(
        len(
            conditions
        )
    ):
        for j in range(
            i + 1,
            len(
                conditions
            ),
        ):
            contrasts.append(
                (
                    conditions[
                        i
                    ],
                    conditions[
                        j
                    ],
                )
            )

    all_rows = []
    summary_rows = []

    for cond_a, cond_b in contrasts:
        mask_a = (
            meta_df[
                condition_col
            ]
            .astype(str)
            == cond_a
        )

        mask_b = (
            meta_df[
                condition_col
            ]
            .astype(str)
            == cond_b
        )

        n_a = int(
            mask_a.sum()
        )

        n_b = int(
            mask_b.sum()
        )

        contrast = (
            f"{cond_a}_vs_{cond_b}"
        )

        if (
            n_a
            < min_samples_per_group
            or n_b
            < min_samples_per_group
        ):
            summary_rows.append(
                {
                    "contrast": contrast,
                    "condition_A": cond_a,
                    "condition_B": cond_b,
                    "n_A": n_a,
                    "n_B": n_b,
                    "status": "skipped_low_sample_count",
                }
            )

            continue

        rows = []

        for gene in genes:
            xa = logcpm_df.loc[
                mask_a,
                gene,
            ].values

            xb = logcpm_df.loc[
                mask_b,
                gene,
            ].values

            mean_a = float(
                np.nanmean(
                    xa
                )
            )

            mean_b = float(
                np.nanmean(
                    xb
                )
            )

            rows.append(
                {
                    "contrast": contrast,
                    "condition_A": cond_a,
                    "condition_B": cond_b,
                    "gene": gene,
                    "n_A": n_a,
                    "n_B": n_b,
                    "mean_logcpm_A": mean_a,
                    "mean_logcpm_B": mean_b,
                    "log2FC_A_vs_B": (
                        mean_a
                        - mean_b
                    ),
                    "pvalue_welch": safe_ttest_ind(
                        xa,
                        xb,
                    ),
                    "pvalue_mannwhitney": safe_mannwhitney(
                        xa,
                        xb,
                    ),
                }
            )

        df = pd.DataFrame(
            rows
        )

        df[
            "padj_welch"
        ] = benjamini_hochberg(
            df[
                "pvalue_welch"
            ].values
        )

        df[
            "padj_mannwhitney"
        ] = benjamini_hochberg(
            df[
                "pvalue_mannwhitney"
            ].values
        )

        all_rows.append(
            df
        )

        summary_rows.append(
            {
                "contrast": contrast,
                "condition_A": cond_a,
                "condition_B": cond_b,
                "n_A": n_a,
                "n_B": n_b,
                "status": "tested",
                "n_genes_tested": int(
                    len(
                        genes
                    )
                ),
                "n_sig_welch_fdr_0_05": int(
                    (
                        df[
                            "padj_welch"
                        ]
                        < 0.05
                    ).sum()
                ),
                "n_sig_mw_fdr_0_05": int(
                    (
                        df[
                            "padj_mannwhitney"
                        ]
                        < 0.05
                    ).sum()
                ),
                "n_sig_abs_log2fc_0_5_fdr_0_05": int(
                    (
                        (
                            df[
                                "padj_welch"
                            ]
                            < 0.05
                        )
                        & (
                            df[
                                "log2FC_A_vs_B"
                            ]
                            .abs()
                            >= 0.5
                        )
                    ).sum()
                ),
            }
        )

    result = (
        pd.concat(
            all_rows,
            ignore_index=True,
        )
        if all_rows
        else pd.DataFrame()
    )

    summary = pd.DataFrame(
        summary_rows
    )

    return result, summary


def extract_top_up_down(
    de_df,
    n_top=100,
    padj_col="padj_welch",
    lfc_col="log2FC_A_vs_B",
):
    rows = []

    for contrast, sub in de_df.groupby(
        "contrast"
    ):
        sub = (
            sub
            .copy()
            .sort_values(
                [
                    padj_col,
                    "pvalue_welch",
                ],
                na_position="last",
            )
        )

        up = (
            sub[
                sub[
                    lfc_col
                ]
                > 0
            ]
            .head(
                n_top
            )
            .copy()
        )

        up[
            "direction"
        ] = "up_in_A"

        down = (
            sub[
                sub[
                    lfc_col
                ]
                < 0
            ]
            .head(
                n_top
            )
            .copy()
        )

        down[
            "direction"
        ] = "up_in_B"

        rows.extend(
            [
                up,
                down,
            ]
        )

    return (
        pd.concat(
            rows,
            ignore_index=True,
        )
        if rows
        else pd.DataFrame()
    )


In [3]:
# =========================
# 2. Load canonical metadata and Notebook-21 raw pseudobulk counts
# =========================
with h5py.File(
    INPUT_H5AD,
    "r",
) as handle:

    obs = read_elem(
        handle[
            "obs"
        ]
    )

    var = read_elem(
        handle[
            "var"
        ]
    )


obs.index = (
    obs.index
    .astype(str)
)

var.index = (
    var.index
    .astype(str)
)


condition_col = choose_first_existing_column(
    obs,
    [
        "condition",
        "disease_condition",
        "diagnosis",
        "group",
    ],
    label="condition column",
)

sample_col = choose_first_existing_column(
    obs,
    [
        "sample_id",
        "sample",
        "sample_name",
        "orig.ident",
        "library_id",
    ],
    label="sample column",
)

patient_col = choose_first_existing_column(
    obs,
    [
        "patient_id",
        "patient",
        "case_id",
        "donor_id",
    ],
    required=False,
    label="patient column",
)

dataset_col = choose_first_existing_column(
    obs,
    [
        "core_dataset",
        "dataset_id",
        "dataset",
        "study_id",
    ],
    required=False,
    label="dataset column",
)

cluster_col = choose_first_existing_column(
    obs,
    [
        "leiden_scvi_main",
        "leiden_scvi_HVG8000_r0_6",
        "leiden",
    ],
    label="cluster column",
)


if (
    obs.shape[
        0
    ],
    var.shape[
        0
    ],
) != (
    351427,
    15176,
):
    raise RuntimeError(
        "Canonical Notebook-20 shape checkpoint failed: "
        f"{(obs.shape[0], var.shape[0])}"
    )


with h5py.File(
    UPSTREAM_PSEUDOBULK_H5AD,
    "r",
) as handle:

    upstream_obs = read_elem(
        handle[
            "obs"
        ]
    )

    upstream_var = read_elem(
        handle[
            "var"
        ]
    )

    if (
        "layers"
        not in handle
        or "counts"
        not in handle[
            "layers"
        ]
    ):
        raise RuntimeError(
            "Notebook-21 pseudobulk H5AD lacks layers['counts']."
        )

    upstream_counts = np.asarray(
        read_elem(
            handle[
                "layers"
            ][
                "counts"
            ]
        )
    )


upstream_obs.index = (
    upstream_obs.index
    .astype(str)
)

upstream_var.index = (
    upstream_var.index
    .astype(str)
)


if upstream_counts.shape != (
    375,
    15176,
):
    raise RuntimeError(
        "Notebook-21 pseudobulk counts shape checkpoint failed: "
        f"{upstream_counts.shape}"
    )


if not np.array_equal(
    upstream_var.index.astype(str),
    var.index.astype(str),
):
    raise RuntimeError(
        "Notebook-21 pseudobulk gene order differs from canonical Notebook-20."
    )


if not np.all(
    np.isfinite(
        upstream_counts
    )
):
    raise RuntimeError(
        "Notebook-21 raw pseudobulk counts contain non-finite values."
    )


if np.any(
    upstream_counts
    < 0
):
    raise RuntimeError(
        "Notebook-21 raw pseudobulk counts contain negative values."
    )


if not np.allclose(
    upstream_counts,
    np.round(
        upstream_counts
    ),
):
    raise RuntimeError(
        "Notebook-21 raw pseudobulk counts are not integer-like."
    )


# Float32 preserves every integer exactly up to 2^24.
# This guards the exact-count reuse assumption explicitly.
max_upstream_count = float(
    np.max(
        upstream_counts
    )
)


if max_upstream_count >= (
    2 ** 24
):
    raise RuntimeError(
        "A Notebook-21 per-gene pseudobulk count exceeds the exact integer "
        "range of float32. Re-aggregate from the raw-cell counts before using "
        "this optimization."
    )


print(
    "condition_col:",
    condition_col,
)

print(
    "sample_col:",
    sample_col,
)

print(
    "patient_col:",
    patient_col,
)

print(
    "dataset_col:",
    dataset_col,
)

print(
    "cluster_col:",
    cluster_col,
)

print(
    "counts_source:",
    "Notebook-21 layers['counts']",
)

print(
    "counts integer-like:",
    True,
)

print(
    "max Notebook-21 per-gene pseudobulk count:",
    max_upstream_count,
)


readiness = pd.DataFrame(
    [
        [
            "input_h5ad",
            str(
                INPUT_H5AD
            ),
        ],
        [
            "n_cells",
            obs.shape[
                0
            ],
        ],
        [
            "n_genes",
            var.shape[
                0
            ],
        ],
        [
            "counts_source",
            "Notebook-21 pseudobulk layers['counts']",
        ],
        [
            "counts_integer_like_sample",
            True,
        ],
        [
            "condition_col",
            condition_col,
        ],
        [
            "sample_col",
            sample_col,
        ],
        [
            "patient_col",
            (
                patient_col
                if patient_col
                else ""
            ),
        ],
        [
            "dataset_col",
            (
                dataset_col
                if dataset_col
                else ""
            ),
        ],
        [
            "cluster_col",
            cluster_col,
        ],
    ],
    columns=[
        "item",
        "value",
    ],
)


write_tsv_replace(
    readiness,
    METADATA_DIR
    / f"{OUT_PREFIX}_readiness.tsv",
)


if readiness.shape != (
    10,
    2,
):
    raise RuntimeError(
        f"Historical readiness checkpoint failed: {readiness.shape}"
    )


print(
    "PASS: canonical metadata and exact Notebook-21 raw pseudobulk counts validated."
)


condition_col: condition
sample_col: sample_id
patient_col: patient_id
dataset_col: core_dataset
cluster_col: leiden_scvi_main
counts_source: Notebook-21 layers['counts']
counts integer-like: True
max Notebook-21 per-gene pseudobulk count: 1784453.0
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_readiness.tsv shape=(10, 2)
PASS: canonical metadata and exact Notebook-21 raw pseudobulk counts validated.


In [4]:
# =========================
# 3. Historical malignant-state definitions and cell counts
# =========================
MALIGNANT_STATE_BY_CLUSTER = {
    "0": "OPC_NPC_like_malignant",
    "4": "hypoxia_stress_malignant",
    "7": "AC_like_reactive_malignant",
    "11": "NPC_neuronal_like_malignant",
    "2": "cycling_malignant_candidate",
}


obs[
    "cluster_for_malignant_state"
] = (
    obs[
        cluster_col
    ]
    .astype(str)
)


obs[
    "condition_malignant_state"
] = (
    obs[
        "cluster_for_malignant_state"
    ]
    .map(
        MALIGNANT_STATE_BY_CLUSTER
    )
    .fillna(
        "not_malignant_state"
    )
)


malig_mask = (
    obs[
        "condition_malignant_state"
    ]
    != "not_malignant_state"
)


print(
    "Malignant/cycling cells selected:",
    int(
        malig_mask.sum()
    ),
)


if int(
    malig_mask.sum()
) != 161884:
    raise RuntimeError(
        "Historical malignant/cycling cell-count checkpoint failed."
    )


state_counts = (
    obs.loc[
        malig_mask,
        "condition_malignant_state",
    ]
    .value_counts()
    .rename_axis(
        "condition_malignant_state"
    )
    .reset_index(
        name="n_cells"
    )
)


state_counts[
    "fraction_of_selected"
] = (
    state_counts[
        "n_cells"
    ]
    / state_counts[
        "n_cells"
    ].sum()
)


write_tsv_replace(
    state_counts,
    METADATA_DIR
    / f"{OUT_PREFIX}_state_cell_counts.tsv",
)


EXPECTED_SELECTED_STATE_COUNTS = {
    "OPC_NPC_like_malignant": 61646,
    "cycling_malignant_candidate": 38634,
    "hypoxia_stress_malignant": 32896,
    "AC_like_reactive_malignant": 22525,
    "NPC_neuronal_like_malignant": 6183,
}


observed_selected_state_counts = {
    str(
        row[
            "condition_malignant_state"
        ]
    ): int(
        row[
            "n_cells"
        ]
    )
    for _, row
    in state_counts.iterrows()
}


if (
    observed_selected_state_counts
    != EXPECTED_SELECTED_STATE_COUNTS
):
    raise RuntimeError(
        "Historical selected-state checkpoint failed: "
        f"{observed_selected_state_counts}"
    )


if state_counts.shape != (
    5,
    3,
):
    raise RuntimeError(
        f"Historical state-count table checkpoint failed: {state_counts.shape}"
    )


display(
    state_counts
)


print(
    "PASS: historical malignant-state selected-cell counts reproduced."
)


Malignant/cycling cells selected: 161884
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_state_cell_counts.tsv shape=(5, 3)


,condition_malignant_state,n_cells,fraction_of_selected
0,OPC_NPC_like_malignant,61646,0.380804
1,cycling_malignant_candidate,38634,0.238652
2,hypoxia_stress_malignant,32896,0.203207
3,AC_like_reactive_malignant,22525,0.139143
4,NPC_neuronal_like_malignant,6183,0.038194


PASS: historical malignant-state selected-cell counts reproduced.


In [5]:
# =========================
# 4. Reconstruct historical sample × state pseudobulks from Notebook 21
# =========================
upstream_state_col = choose_first_existing_column(
    upstream_obs,
    [
        "malignant_state_for_pseudobulk",
        "condition_malignant_state",
        "malignant_state",
    ],
    label="Notebook-21 malignant-state column",
)

upstream_sample_col = choose_first_existing_column(
    upstream_obs,
    [
        "sample_id",
        sample_col,
    ],
    label="Notebook-21 sample column",
)

upstream_condition_col = choose_first_existing_column(
    upstream_obs,
    [
        "condition",
        condition_col,
    ],
    label="Notebook-21 condition column",
)

upstream_patient_col = choose_first_existing_column(
    upstream_obs,
    [
        "patient_id",
        patient_col,
    ] if patient_col else [
        "patient_id",
    ],
    required=False,
    label="Notebook-21 patient column",
)


state_counts_all = pd.DataFrame(
    upstream_counts.astype(
        np.float64,
        copy=False,
    ),
    columns=var.index,
)


state_meta_all = pd.DataFrame(
    {
        "sample_id": upstream_obs[
            upstream_sample_col
        ]
        .astype(str)
        .to_numpy(),
        "condition_malignant_state": upstream_obs[
            upstream_state_col
        ]
        .astype(str)
        .to_numpy(),
        "condition": upstream_obs[
            upstream_condition_col
        ]
        .astype(str)
        .to_numpy(),
        "n_cells": pd.to_numeric(
            upstream_obs[
                "n_cells"
            ],
            errors="raise",
        )
        .astype(int)
        .to_numpy(),
    }
)


if (
    upstream_patient_col
    and upstream_patient_col
    in upstream_obs.columns
):
    state_meta_all[
        "patient_id"
    ] = (
        upstream_obs[
            upstream_patient_col
        ]
        .astype(str)
        .to_numpy()
    )


# Recover canonical sample metadata from the Notebook-20 obs.
sample_meta_cols = [
    sample_col,
    condition_col,
]

if patient_col:
    sample_meta_cols.append(
        patient_col
    )

if dataset_col:
    sample_meta_cols.append(
        dataset_col
    )


sample_meta_source = (
    obs.loc[
        malig_mask,
        sample_meta_cols,
    ]
    .copy()
)


for column in sample_meta_cols:
    sample_meta_source[
        column
    ] = (
        sample_meta_source[
            column
        ]
        .astype(str)
    )


# Each sample must have one condition/patient/dataset identity.
for column in sample_meta_cols[
    1:
]:
    max_unique = (
        sample_meta_source
        .groupby(
            sample_col,
            observed=True,
        )[
            column
        ]
        .nunique(
            dropna=False
        )
        .max()
    )

    if int(
        max_unique
    ) != 1:
        raise RuntimeError(
            f"Sample metadata is not unique for {column}."
        )


sample_meta_map = (
    sample_meta_source
    .drop_duplicates(
        subset=[
            sample_col
        ]
    )
    .set_index(
        sample_col
    )
)


if patient_col:
    if "patient_id" not in state_meta_all.columns:
        state_meta_all[
            "patient_id"
        ] = (
            state_meta_all[
                "sample_id"
            ]
            .map(
                sample_meta_map[
                    patient_col
                ]
            )
        )


if dataset_col:
    state_meta_all[
        "dataset_id"
    ] = (
        state_meta_all[
            "sample_id"
        ]
        .map(
            sample_meta_map[
                dataset_col
            ]
        )
    )


# Restore the exact historical group-key order: first appearance among selected cells.
obs_sel_order = (
    obs.loc[
        malig_mask,
        [
            sample_col,
            "condition_malignant_state",
        ],
    ]
    .copy()
)

obs_sel_order[
    "group_key"
] = (
    obs_sel_order[
        sample_col
    ]
    .astype(str)
    + "||"
    + obs_sel_order[
        "condition_malignant_state"
    ]
    .astype(str)
)


historical_state_order = pd.Index(
    obs_sel_order[
        "group_key"
    ]
    .drop_duplicates()
    .tolist()
)


state_meta_all[
    "group_key"
] = (
    state_meta_all[
        "sample_id"
    ]
    .astype(str)
    + "||"
    + state_meta_all[
        "condition_malignant_state"
    ]
    .astype(str)
)


if set(
    state_meta_all[
        "group_key"
    ]
) != set(
    historical_state_order
):
    raise RuntimeError(
        "Notebook-21 sample-state groups do not match the exact historical "
        "Notebook-22 selected-cell groups."
    )


state_counts_all.index = (
    state_meta_all[
        "group_key"
    ]
    .astype(str)
    .to_numpy()
)

state_meta_all = (
    state_meta_all
    .set_index(
        "group_key"
    )
)


state_counts_all = (
    state_counts_all.loc[
        historical_state_order
    ]
    .copy()
)

state_meta_all = (
    state_meta_all.loc[
        historical_state_order
    ]
    .copy()
)


# Recompute historical QC metrics from raw counts.
state_library_size = (
    state_counts_all
    .sum(
        axis=1
    )
)

state_n_genes_detected = (
    state_counts_all
    .gt(
        0
    )
    .sum(
        axis=1
    )
)


state_meta_all[
    "library_size"
] = (
    state_library_size
    .astype(float)
)

state_meta_all[
    "n_genes_detected"
] = (
    state_n_genes_detected
    .astype(int)
)


if state_counts_all.shape != (
    375,
    15176,
):
    raise RuntimeError(
        f"Historical all sample-state count shape failed: {state_counts_all.shape}"
    )


if state_meta_all.shape[
    0
] != 375:
    raise RuntimeError(
        "Historical sample-state metadata row-count checkpoint failed."
    )


print(
    "PASS: exact historical 375 sample-state pseudobulk groups reconstructed."
)


PASS: exact historical 375 sample-state pseudobulk groups reconstructed.


In [6]:
# =========================
# 5. Overall malignant/cycling sample pseudobulks
# =========================
historical_sample_order = pd.Index(
    obs.loc[
        malig_mask,
        sample_col,
    ]
    .astype(str)
    .drop_duplicates()
    .tolist()
)


sample_position = {
    sample: index
    for index, sample
    in enumerate(
        historical_sample_order
    )
}


overall_counts_array = np.zeros(
    (
        len(
            historical_sample_order
        ),
        var.shape[
            0
        ],
    ),
    dtype=np.float64,
)


overall_n_cells = np.zeros(
    len(
        historical_sample_order
    ),
    dtype=np.int64,
)


for group_key, row_meta in state_meta_all.iterrows():
    sample = str(
        row_meta[
            "sample_id"
        ]
    )

    position = sample_position[
        sample
    ]

    overall_counts_array[
        position,
        :,
    ] += state_counts_all.loc[
        group_key
    ].to_numpy(
        dtype=np.float64,
        copy=False,
    )

    overall_n_cells[
        position
    ] += int(
        row_meta[
            "n_cells"
        ]
    )


pb_counts = pd.DataFrame(
    overall_counts_array,
    index=historical_sample_order,
    columns=var.index,
)


pb_meta = pd.DataFrame(
    index=historical_sample_order
)


pb_meta[
    "sample_id"
] = (
    historical_sample_order
    .astype(str)
)


pb_meta[
    "condition"
] = (
    pb_meta[
        "sample_id"
    ]
    .map(
        sample_meta_map[
            condition_col
        ]
    )
    .astype(str)
)


if patient_col:
    pb_meta[
        "patient_id"
    ] = (
        pb_meta[
            "sample_id"
        ]
        .map(
            sample_meta_map[
                patient_col
            ]
        )
        .astype(str)
    )


if dataset_col:
    pb_meta[
        "dataset_id"
    ] = (
        pb_meta[
            "sample_id"
        ]
        .map(
            sample_meta_map[
                dataset_col
            ]
        )
        .astype(str)
    )


pb_meta[
    "n_cells"
] = overall_n_cells

pb_meta[
    "library_size"
] = (
    pb_counts
    .sum(
        axis=1
    )
    .astype(float)
)

pb_meta[
    "n_genes_detected"
] = (
    pb_counts
    .gt(
        0
    )
    .sum(
        axis=1
    )
    .astype(int)
)


SAMPLE_MIN_CELLS = 30


keep_samples = (
    pb_meta[
        "n_cells"
    ]
    >= SAMPLE_MIN_CELLS
)


pb_counts_f = (
    pb_counts.loc[
        keep_samples
    ]
    .copy()
)

pb_meta_f = (
    pb_meta.loc[
        keep_samples
    ]
    .copy()
)


gene_keep = filter_genes_by_cpm(
    pb_counts_f,
    min_cpm=1.0,
    min_samples=3,
)


genes_test = (
    gene_keep[
        gene_keep
    ]
    .index
    .tolist()
)


print(
    "Pseudobulk samples:",
    pb_counts.shape[
        0
    ],
)

print(
    "Pseudobulk samples passing:",
    pb_counts_f.shape[
        0
    ],
)

print(
    "Genes passing expression filter:",
    len(
        genes_test
    ),
)


if pb_counts.shape[
    0
] != 80:
    raise RuntimeError(
        "Historical overall malignant pseudobulk sample count failed."
    )


if pb_counts_f.shape[
    0
] != 77:
    raise RuntimeError(
        "Historical passing overall malignant pseudobulk count failed."
    )


if len(
    genes_test
) != 15081:
    raise RuntimeError(
        "Historical overall malignant gene-filter checkpoint failed: "
        f"{len(genes_test)} != 15081"
    )


sample_qc = (
    pb_meta_f
    .reset_index(
        names="group_key"
    )
)


sample_qc[
    "n_genes_total"
] = var.shape[
    0
]

sample_qc[
    "n_genes_tested"
] = len(
    genes_test
)


write_tsv_replace(
    sample_qc,
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_sample_QC.tsv",
)


if sample_qc.shape != (
    77,
    10,
):
    raise RuntimeError(
        f"Historical overall sample QC shape failed: {sample_qc.shape}"
    )


overall_counts_path = (
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_counts_pseudobulk.tsv.gz"
)

overall_logcpm_path = (
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_logCPM_pseudobulk.tsv.gz"
)


write_gzip_tsv_replace(
    pb_counts_f[
        genes_test
    ],
    overall_counts_path,
    index=True,
)


logcpm_overall = counts_to_logcpm(
    pb_counts_f[
        genes_test
    ]
)


write_gzip_tsv_replace(
    logcpm_overall,
    overall_logcpm_path,
    index=True,
)


condition_summary = (
    pb_meta_f
    .groupby(
        "condition"
    )
    .agg(
        n_samples=(
            "sample_id",
            "nunique",
        ),
        n_pseudobulks=(
            "sample_id",
            "size",
        ),
        median_cells=(
            "n_cells",
            "median",
        ),
        total_cells=(
            "n_cells",
            "sum",
        ),
        median_library_size=(
            "library_size",
            "median",
        ),
    )
    .reset_index()
)


write_tsv_replace(
    condition_summary,
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_condition_summary.tsv",
)


if condition_summary.shape != (
    4,
    6,
):
    raise RuntimeError(
        f"Historical condition-summary shape failed: {condition_summary.shape}"
    )


EXPECTED_OVERALL_CONDITION_SUMMARY = {
    "GBM": (
        39,
        39,
        1162.0,
        82869,
        4191294.0,
    ),
    "LGG": (
        4,
        4,
        1510.0,
        5023,
        16806812.5,
    ),
    "ndGBM": (
        18,
        18,
        1800.0,
        45136,
        16141893.5,
    ),
    "rGBM": (
        16,
        16,
        1412.0,
        28835,
        15412161.0,
    ),
}


for condition, expected in EXPECTED_OVERALL_CONDITION_SUMMARY.items():
    row = condition_summary.loc[
        condition_summary[
            "condition"
        ]
        .astype(str)
        .eq(
            condition
        )
    ]

    if len(
        row
    ) != 1:
        raise RuntimeError(
            f"Historical overall condition row missing: {condition}"
        )

    row = row.iloc[
        0
    ]

    observed = (
        int(
            row[
                "n_samples"
            ]
        ),
        int(
            row[
                "n_pseudobulks"
            ]
        ),
        float(
            row[
                "median_cells"
            ]
        ),
        int(
            row[
                "total_cells"
            ]
        ),
        float(
            row[
                "median_library_size"
            ]
        ),
    )

    if not all(
        np.isclose(
            float(
                observed_value
            ),
            float(
                expected_value
            ),
            atol=1e-6,
            rtol=0,
        )
        for observed_value, expected_value
        in zip(
            observed,
            expected,
        )
    ):
        raise RuntimeError(
            f"Historical overall condition checkpoint failed for {condition}: "
            f"{observed} != {expected}"
        )


display(
    condition_summary
)


print(
    "PASS: historical overall malignant sample pseudobulks reproduced exactly."
)


Pseudobulk samples: 80
Pseudobulk samples passing: 77
Genes passing expression filter: 15081
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_sample_QC.tsv shape=(77, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_counts_pseudobulk.tsv.gz shape=(77, 15081)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_logCPM_pseudobulk.tsv.gz shape=(77, 15081)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_condition_summary.tsv shape=(4, 6)


,condition,n_samples,n_pseudobulks,median_cells,total_cells,median_library_size
0,GBM,39,39,1162.0,82869,4191294.0
1,LGG,4,4,1510.0,5023,16806812.5
2,ndGBM,18,18,1800.0,45136,16141893.5
3,rGBM,16,16,1412.0,28835,15412161.0


PASS: historical overall malignant sample pseudobulks reproduced exactly.


In [7]:
# =========================
# 6. Overall malignant/cycling condition DE
# =========================
global_kruskal = run_global_kruskal(
    logcpm_overall,
    pb_meta_f,
    "condition",
    genes_test,
)


write_tsv_replace(
    global_kruskal,
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_global_kruskal.tsv",
)


pairwise_de, pairwise_summary = run_pairwise_de(
    logcpm_overall,
    pb_meta_f,
    "condition",
    genes_test,
    min_samples_per_group=3,
)


write_tsv_replace(
    pairwise_summary,
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_test_summary.tsv",
)


write_tsv_replace(
    pairwise_de,
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_all_genes.tsv",
)


top100_overall = extract_top_up_down(
    pairwise_de,
    n_top=100,
)


write_tsv_replace(
    top100_overall,
    METADATA_DIR
    / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_top100_up_down.tsv",
)


sig_overall = pairwise_de[
    (
        pairwise_de[
            "padj_welch"
        ]
        < 0.05
    )
    & (
        pairwise_de[
            "log2FC_A_vs_B"
        ]
        .abs()
        >= 0.5
    )
].copy()


write_tsv_replace(
    sig_overall,
    METADATA_DIR
    / (
        f"{OUT_PREFIX}_overall_malignant_pairwise_DE_"
        "significant_FDR0_05_absLFC0_5.tsv"
    ),
)


EXPECTED_OVERALL_SHAPES = {
    "global_kruskal": (
        15081,
        9,
    ),
    "pairwise_summary": (
        6,
        10,
    ),
    "pairwise_de": (
        90486,
        13,
    ),
    "top100_overall": (
        1200,
        14,
    ),
    "sig_overall": (
        27004,
        13,
    ),
}


observed_overall_shapes = {
    "global_kruskal": global_kruskal.shape,
    "pairwise_summary": pairwise_summary.shape,
    "pairwise_de": pairwise_de.shape,
    "top100_overall": top100_overall.shape,
    "sig_overall": sig_overall.shape,
}


if observed_overall_shapes != EXPECTED_OVERALL_SHAPES:
    raise RuntimeError(
        "Historical overall-condition DE shape checkpoint failed. "
        f"Observed: {observed_overall_shapes}"
    )


EXPECTED_OVERALL_DE_COUNTS = {
    "GBM_vs_LGG": (
        39,
        4,
        8504,
        6015,
        7455,
    ),
    "GBM_vs_ndGBM": (
        39,
        18,
        4584,
        4192,
        3308,
    ),
    "GBM_vs_rGBM": (
        39,
        16,
        5285,
        4556,
        4089,
    ),
    "LGG_vs_ndGBM": (
        4,
        18,
        6164,
        4569,
        5512,
    ),
    "LGG_vs_rGBM": (
        4,
        16,
        6553,
        5270,
        5970,
    ),
    "ndGBM_vs_rGBM": (
        18,
        16,
        919,
        862,
        670,
    ),
}


for contrast, expected in EXPECTED_OVERALL_DE_COUNTS.items():
    row = pairwise_summary.loc[
        pairwise_summary[
            "contrast"
        ]
        .astype(str)
        .eq(
            contrast
        )
    ]

    if len(
        row
    ) != 1:
        raise RuntimeError(
            f"Historical overall DE contrast missing/non-unique: {contrast}"
        )

    row = row.iloc[
        0
    ]

    observed = (
        int(
            row[
                "n_A"
            ]
        ),
        int(
            row[
                "n_B"
            ]
        ),
        int(
            row[
                "n_sig_welch_fdr_0_05"
            ]
        ),
        int(
            row[
                "n_sig_mw_fdr_0_05"
            ]
        ),
        int(
            row[
                "n_sig_abs_log2fc_0_5_fdr_0_05"
            ]
        ),
    )

    if observed != expected:
        raise RuntimeError(
            f"Historical overall DE checkpoint failed for {contrast}: "
            f"{observed} != {expected}"
        )


display(
    pairwise_summary
)

display(
    top100_overall.head(
        20
    )
)


print(
    "PASS: historical overall malignant/cycling condition-DE outputs reproduced."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_global_kruskal.tsv shape=(15081, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_test_summary.tsv shape=(6, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_all_genes.tsv shape=(90486, 13)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_top100_up_down.tsv shape=(1200, 14)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv shape=(27004, 13)


,contrast,condition_A,condition_B,n_A,n_B,status,n_genes_tested,n_sig_welch_fdr_0_05,n_sig_mw_fdr_0_05,n_sig_abs_log2fc_0_5_fdr_0_05
0,GBM_vs_LGG,GBM,LGG,39,4,tested,15081,8504,6015,7455
1,GBM_vs_ndGBM,GBM,ndGBM,39,18,tested,15081,4584,4192,3308
2,GBM_vs_rGBM,GBM,rGBM,39,16,tested,15081,5285,4556,4089
3,LGG_vs_ndGBM,LGG,ndGBM,4,18,tested,15081,6164,4569,5512
4,LGG_vs_rGBM,LGG,rGBM,4,16,tested,15081,6553,5270,5970
5,ndGBM_vs_rGBM,ndGBM,rGBM,18,16,tested,15081,919,862,670


,contrast,condition_A,condition_B,gene,n_A,n_B,mean_logcpm_A,mean_logcpm_B,log2FC_A_vs_B,pvalue_welch,pvalue_mannwhitney,padj_welch,padj_mannwhitney,direction
0,GBM_vs_LGG,GBM,LGG,METTL13,39,4,4.613037,0.000000,4.613037,5.781682e-36,0.001188,8.719354e-32,0.006439,up_in_A
1,GBM_vs_LGG,GBM,LGG,UTP14C,39,4,4.160341,0.000000,4.160341,2.596442e-32,0.001188,1.957847e-28,0.006439,up_in_A
2,GBM_vs_LGG,GBM,LGG,LINC01579,39,4,4.537858,0.031680,4.506178,4.703016e-28,0.001191,1.182103e-24,0.006439,up_in_A
3,GBM_vs_LGG,GBM,LGG,PDPN,39,4,6.928661,0.155358,6.773303,9.218364e-25,0.000032,1.390221e-21,0.000568,up_in_A
4,GBM_vs_LGG,GBM,LGG,TIMP1,39,4,8.769671,0.347938,8.421733,1.320116e-24,0.000113,1.809878e-21,0.001293,up_in_A
5,GBM_vs_LGG,GBM,LGG,C5orf63,39,4,3.532873,0.191481,3.341391,1.454761e-24,0.000016,1.828271e-21,0.000403,up_in_A
6,GBM_vs_LGG,GBM,LGG,FBXO17,39,4,5.031271,0.349951,4.681320,1.960948e-24,0.000032,2.274850e-21,0.000568,up_in_A
7,GBM_vs_LGG,GBM,LGG,ST20,39,4,4.107645,0.000000,4.107645,3.297145e-24,0.001188,3.551732e-21,0.006439,up_in_A
8,GBM_vs_LGG,GBM,LGG,ZIM2-AS1,39,4,3.663298,0.000000,3.663298,3.871145e-24,0.001188,3.892049e-21,0.006439,up_in_A
9,GBM_vs_LGG,GBM,LGG,PLAT,39,4,5.524516,0.224770,5.299746,2.699311e-23,0.000016,2.544269e-20,0.000403,up_in_A


PASS: historical overall malignant/cycling condition-DE outputs reproduced.


In [8]:
# =========================
# 7. Historical sample × state pseudobulk QC
# =========================
MIN_CELLS_PER_SAMPLE_STATE = 30


keep_state_pb = (
    state_meta_all[
        "n_cells"
    ]
    >= MIN_CELLS_PER_SAMPLE_STATE
)


state_counts_f = (
    state_counts_all.loc[
        keep_state_pb
    ]
    .copy()
)


state_meta_f = (
    state_meta_all.loc[
        keep_state_pb
    ]
    .copy()
)


state_gene_keep = filter_genes_by_cpm(
    state_counts_f,
    min_cpm=1.0,
    min_samples=3,
)


state_genes_test = (
    state_gene_keep[
        state_gene_keep
    ]
    .index
    .tolist()
)


if state_counts_f.shape[
    0
] != 290:
    raise RuntimeError(
        "Historical passing sample-state pseudobulk count failed."
    )


if len(
    state_genes_test
) != 15176:
    raise RuntimeError(
        "Historical within-state expression-filter checkpoint failed: "
        f"{len(state_genes_test)} != 15176"
    )


logcpm_state = counts_to_logcpm(
    state_counts_f[
        state_genes_test
    ]
)


state_pb_qc = (
    state_meta_f
    .reset_index(
        names="group_key"
    )
)


state_pb_qc[
    "n_genes_tested"
] = len(
    state_genes_test
)


write_tsv_replace(
    state_pb_qc,
    METADATA_DIR
    / f"{OUT_PREFIX}_sample_state_pseudobulk_QC.tsv",
)


if state_pb_qc.shape != (
    290,
    10,
):
    raise RuntimeError(
        f"Historical sample-state QC shape failed: {state_pb_qc.shape}"
    )


state_condition_summary = (
    state_meta_f
    .groupby(
        [
            "condition_malignant_state",
            "condition",
        ]
    )
    .agg(
        n_pseudobulks=(
            "sample_id",
            "size",
        ),
        n_samples=(
            "sample_id",
            "nunique",
        ),
        total_cells=(
            "n_cells",
            "sum",
        ),
        median_cells=(
            "n_cells",
            "median",
        ),
    )
    .reset_index()
)


write_tsv_replace(
    state_condition_summary,
    METADATA_DIR
    / f"{OUT_PREFIX}_state_condition_pseudobulk_summary.tsv",
)


if state_condition_summary.shape != (
    19,
    6,
):
    raise RuntimeError(
        "Historical state-condition summary shape checkpoint failed: "
        f"{state_condition_summary.shape}"
    )


EXPECTED_STATE_CONDITION = {
    (
        "AC_like_reactive_malignant",
        "GBM",
    ): (
        34,
        34,
        13733,
        172.5,
    ),
    (
        "AC_like_reactive_malignant",
        "LGG",
    ): (
        2,
        2,
        472,
        236.0,
    ),
    (
        "AC_like_reactive_malignant",
        "ndGBM",
    ): (
        17,
        17,
        4845,
        296.0,
    ),
    (
        "AC_like_reactive_malignant",
        "rGBM",
    ): (
        13,
        13,
        3298,
        219.0,
    ),
    (
        "NPC_neuronal_like_malignant",
        "GBM",
    ): (
        17,
        17,
        4236,
        131.0,
    ),
    (
        "NPC_neuronal_like_malignant",
        "LGG",
    ): (
        3,
        3,
        218,
        41.0,
    ),
    (
        "NPC_neuronal_like_malignant",
        "ndGBM",
    ): (
        7,
        7,
        1167,
        111.0,
    ),
    (
        "NPC_neuronal_like_malignant",
        "rGBM",
    ): (
        4,
        4,
        247,
        58.0,
    ),
    (
        "OPC_NPC_like_malignant",
        "GBM",
    ): (
        36,
        36,
        27325,
        479.5,
    ),
    (
        "OPC_NPC_like_malignant",
        "LGG",
    ): (
        4,
        4,
        3975,
        1115.5,
    ),
    (
        "OPC_NPC_like_malignant",
        "ndGBM",
    ): (
        15,
        15,
        18782,
        845.0,
    ),
    (
        "OPC_NPC_like_malignant",
        "rGBM",
    ): (
        14,
        14,
        11423,
        689.5,
    ),
    (
        "cycling_malignant_candidate",
        "GBM",
    ): (
        32,
        32,
        17221,
        358.5,
    ),
    (
        "cycling_malignant_candidate",
        "LGG",
    ): (
        4,
        4,
        283,
        64.5,
    ),
    (
        "cycling_malignant_candidate",
        "ndGBM",
    ): (
        16,
        16,
        10730,
        449.0,
    ),
    (
        "cycling_malignant_candidate",
        "rGBM",
    ): (
        14,
        14,
        10309,
        299.5,
    ),
    (
        "hypoxia_stress_malignant",
        "GBM",
    ): (
        31,
        31,
        19859,
        215.0,
    ),
    (
        "hypoxia_stress_malignant",
        "ndGBM",
    ): (
        15,
        15,
        9463,
        261.0,
    ),
    (
        "hypoxia_stress_malignant",
        "rGBM",
    ): (
        12,
        12,
        3363,
        205.0,
    ),
}


for key, expected in EXPECTED_STATE_CONDITION.items():
    state, condition = key

    row = state_condition_summary.loc[
        state_condition_summary[
            "condition_malignant_state"
        ]
        .astype(str)
        .eq(
            state
        )
        & state_condition_summary[
            "condition"
        ]
        .astype(str)
        .eq(
            condition
        )
    ]

    if len(
        row
    ) != 1:
        raise RuntimeError(
            f"Historical state-condition row missing/non-unique: {key}"
        )

    row = row.iloc[
        0
    ]

    observed = (
        int(
            row[
                "n_pseudobulks"
            ]
        ),
        int(
            row[
                "n_samples"
            ]
        ),
        int(
            row[
                "total_cells"
            ]
        ),
        float(
            row[
                "median_cells"
            ]
        ),
    )

    if not all(
        np.isclose(
            float(
                observed_value
            ),
            float(
                expected_value
            ),
            atol=1e-6,
            rtol=0,
        )
        for observed_value, expected_value
        in zip(
            observed,
            expected,
        )
    ):
        raise RuntimeError(
            f"Historical state-condition checkpoint failed for {key}: "
            f"{observed} != {expected}"
        )


display(
    state_condition_summary
)


print(
    "PASS: historical sample-state pseudobulk QC reproduced exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_sample_state_pseudobulk_QC.tsv shape=(290, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_state_condition_pseudobulk_summary.tsv shape=(19, 6)


,condition_malignant_state,condition,n_pseudobulks,n_samples,total_cells,median_cells
0,AC_like_reactive_malignant,GBM,34,34,13733,172.5
1,AC_like_reactive_malignant,LGG,2,2,472,236.0
2,AC_like_reactive_malignant,ndGBM,17,17,4845,296.0
3,AC_like_reactive_malignant,rGBM,13,13,3298,219.0
4,NPC_neuronal_like_malignant,GBM,17,17,4236,131.0
5,NPC_neuronal_like_malignant,LGG,3,3,218,41.0
6,NPC_neuronal_like_malignant,ndGBM,7,7,1167,111.0
7,NPC_neuronal_like_malignant,rGBM,4,4,247,58.0
8,OPC_NPC_like_malignant,GBM,36,36,27325,479.5
9,OPC_NPC_like_malignant,LGG,4,4,3975,1115.5


PASS: historical sample-state pseudobulk QC reproduced exactly.


In [9]:
# =========================
# 8. Within-state condition DE
# =========================
within_rows = []
within_summary_rows = []
top_rows = []


# Historical v3 duplicate-state-column fix.
def add_state_column(
    df,
    state,
    col="condition_malignant_state",
):
    df = df.copy()

    if len(
        df
    ) == 0:
        return df

    if col in df.columns:
        df[
            col
        ] = df[
            col
        ].astype(str)

        empty_mask = df[
            col
        ].isin(
            [
                "",
                "nan",
                "None",
                "NaN",
            ]
        )

        if empty_mask.any():
            df.loc[
                empty_mask,
                col,
            ] = str(
                state
            )

    else:
        df.insert(
            0,
            col,
            str(
                state
            ),
        )

    return df


for state, idx in state_meta_f.groupby(
    "condition_malignant_state"
).groups.items():
    state_idx = list(
        idx
    )

    meta_s = (
        state_meta_f.loc[
            state_idx
        ]
        .copy()
    )

    log_s = (
        logcpm_state.loc[
            state_idx,
            :,
        ]
        .copy()
    )


    de_s, summary_s = run_pairwise_de(
        log_s,
        meta_s,
        "condition",
        state_genes_test,
        min_samples_per_group=3,
    )


    if len(
        summary_s
    ) > 0:
        summary_s = add_state_column(
            summary_s,
            state,
        )

        within_summary_rows.append(
            summary_s
        )


    if len(
        de_s
    ) > 0:
        de_s = add_state_column(
            de_s,
            state,
        )

        within_rows.append(
            de_s
        )


        top_s = extract_top_up_down(
            de_s,
            n_top=50,
        )

        if len(
            top_s
        ) > 0:
            top_s = add_state_column(
                top_s,
                state,
            )

            top_rows.append(
                top_s
            )


within_state_de = (
    pd.concat(
        within_rows,
        ignore_index=True,
    )
    if within_rows
    else pd.DataFrame()
)


within_state_summary = (
    pd.concat(
        within_summary_rows,
        ignore_index=True,
    )
    if within_summary_rows
    else pd.DataFrame()
)


within_state_top50 = (
    pd.concat(
        top_rows,
        ignore_index=True,
    )
    if top_rows
    else pd.DataFrame()
)


write_tsv_replace(
    within_state_summary,
    METADATA_DIR
    / f"{OUT_PREFIX}_within_state_condition_DE_test_summary.tsv",
)


write_tsv_replace(
    within_state_de,
    METADATA_DIR
    / f"{OUT_PREFIX}_within_state_condition_DE_all_genes.tsv",
)


write_tsv_replace(
    within_state_top50,
    METADATA_DIR
    / f"{OUT_PREFIX}_within_state_condition_DE_top50_up_down.tsv",
)


EXPECTED_WITHIN_SHAPES = {
    "summary": (
        27,
        11,
    ),
    "all_genes": (
        364224,
        14,
    ),
    "top50": (
        2400,
        15,
    ),
}


observed_within_shapes = {
    "summary": within_state_summary.shape,
    "all_genes": within_state_de.shape,
    "top50": within_state_top50.shape,
}


if observed_within_shapes != EXPECTED_WITHIN_SHAPES:
    raise RuntimeError(
        "Historical within-state DE shape checkpoint failed. "
        f"Observed: {observed_within_shapes}"
    )


if int(
    (
        within_state_summary[
            "status"
        ]
        .astype(str)
        == "tested"
    ).sum()
) != 24:
    raise RuntimeError(
        "Historical within-state tested-contrast checkpoint failed."
    )


if int(
    (
        within_state_summary[
            "status"
        ]
        .astype(str)
        == "skipped_low_sample_count"
    ).sum()
) != 3:
    raise RuntimeError(
        "Historical within-state skipped-contrast checkpoint failed."
    )


tested_gene_counts = set(
    pd.to_numeric(
        within_state_summary.loc[
            within_state_summary[
                "status"
            ]
            .astype(str)
            .eq(
                "tested"
            ),
            "n_genes_tested",
        ],
        errors="coerce",
    )
    .dropna()
    .astype(int)
    .tolist()
)


if tested_gene_counts != {
    15176
}:
    raise RuntimeError(
        "Historical within-state genes-tested checkpoint failed: "
        f"{tested_gene_counts}"
    )


display(
    within_state_summary.head(
        30
    )
)


display(
    within_state_top50.head(
        20
    )
)


print(
    "PASS: historical within-state condition-DE structure reproduced."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_within_state_condition_DE_test_summary.tsv shape=(27, 11)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_within_state_condition_DE_all_genes.tsv shape=(364224, 14)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_within_state_condition_DE_top50_up_down.tsv shape=(2400, 15)


,condition_malignant_state,contrast,condition_A,condition_B,n_A,n_B,status,n_genes_tested,n_sig_welch_fdr_0_05,n_sig_mw_fdr_0_05,n_sig_abs_log2fc_0_5_fdr_0_05
0,AC_like_reactive_malignant,GBM_vs_LGG,GBM,LGG,34,2,skipped_low_sample_count,NaN,NaN,NaN,NaN
1,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,34,17,tested,15176.0,3664.0,3557.0,3254.0
2,AC_like_reactive_malignant,GBM_vs_rGBM,GBM,rGBM,34,13,tested,15176.0,4324.0,3460.0,3994.0
3,AC_like_reactive_malignant,LGG_vs_ndGBM,LGG,ndGBM,2,17,skipped_low_sample_count,NaN,NaN,NaN,NaN
4,AC_like_reactive_malignant,LGG_vs_rGBM,LGG,rGBM,2,13,skipped_low_sample_count,NaN,NaN,NaN,NaN
5,AC_like_reactive_malignant,ndGBM_vs_rGBM,ndGBM,rGBM,17,13,tested,15176.0,662.0,394.0,582.0
6,NPC_neuronal_like_malignant,GBM_vs_LGG,GBM,LGG,17,3,tested,15176.0,4184.0,759.0,4127.0
7,NPC_neuronal_like_malignant,GBM_vs_ndGBM,GBM,ndGBM,17,7,tested,15176.0,585.0,593.0,582.0
8,NPC_neuronal_like_malignant,GBM_vs_rGBM,GBM,rGBM,17,4,tested,15176.0,1235.0,119.0,1219.0
9,NPC_neuronal_like_malignant,LGG_vs_ndGBM,LGG,ndGBM,3,7,tested,15176.0,1050.0,0.0,1046.0


,condition_malignant_state,contrast,condition_A,condition_B,gene,n_A,n_B,mean_logcpm_A,mean_logcpm_B,log2FC_A_vs_B,pvalue_welch,pvalue_mannwhitney,padj_welch,padj_mannwhitney,direction
0,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,ALDOA,34,17,9.920246,4.873269,5.046976,4.504157e-21,8.184096e-09,3.417755e-17,0.000010,up_in_A
1,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,MKKS,34,17,6.539089,4.795984,1.743105,4.461280e-18,4.137428e-08,1.745186e-14,0.000015,up_in_A
2,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,NACA2,34,17,5.156555,0.433608,4.722947,2.325979e-16,6.892180e-09,7.059812e-13,0.000010,up_in_A
3,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,UQCRHL,34,17,6.046584,2.034715,4.011868,4.590560e-15,1.165631e-08,6.333304e-12,0.000010,up_in_A
4,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,IMPDH2,34,17,6.880199,5.770178,1.110021,4.189507e-14,2.339828e-08,4.890766e-11,0.000011,up_in_A
5,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,HSPA1B,34,17,10.158097,7.393167,2.764931,8.905848e-14,6.482368e-08,8.447197e-11,0.000018,up_in_A
6,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,UROD,34,17,7.208154,5.782494,1.425660,2.134246e-13,2.085301e-08,1.799407e-10,0.000011,up_in_A
7,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,FKBP1C,34,17,4.466456,0.198027,4.268429,6.820386e-13,9.277782e-08,5.175309e-10,0.000020,up_in_A
8,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,WDR1,34,17,7.209711,6.198756,1.010954,1.134318e-12,1.126415e-07,8.090953e-10,0.000022,up_in_A
9,AC_like_reactive_malignant,GBM_vs_ndGBM,GBM,ndGBM,RNASEH2A,34,17,4.823705,2.302768,2.520937,1.172911e-12,1.310450e-08,8.090953e-10,0.000010,up_in_A


PASS: historical within-state condition-DE structure reproduced.


In [10]:
# =========================
# 9. Historical program scores
# =========================
PROGRAMS = {
    "OPC_proneural": [
        "OLIG1",
        "OLIG2",
        "SOX10",
        "PDGFRA",
        "BCAN",
        "PTPRZ1",
        "PLP1",
        "CSPG4",
        "DLL3",
    ],
    "NPC_neural": [
        "SOX11",
        "DCX",
        "STMN2",
        "TUBB3",
        "RBFOX2",
        "ELAVL4",
        "DLX5",
        "MYT1L",
        "SLC17A6",
    ],
    "AC_reactive": [
        "GFAP",
        "AQP4",
        "CLU",
        "SPARCL1",
        "ID3",
        "ID4",
        "FAM107A",
        "SLC1A3",
    ],
    "MES_like": [
        "CHI3L1",
        "CD44",
        "VIM",
        "TGFBI",
        "SERPINE1",
        "ANXA1",
        "LGALS3",
        "FN1",
        "CTSB",
    ],
    "hypoxia_stress": [
        "NDRG1",
        "VEGFA",
        "SLC2A1",
        "HILPDA",
        "ADM",
        "EGLN3",
        "P4HA1",
        "PGK1",
        "CA9",
    ],
    "cycling": [
        "TOP2A",
        "MKI67",
        "UBE2C",
        "PBK",
        "ASPM",
        "RRM2",
        "CENPF",
        "DLGAP5",
        "BIRC5",
    ],
    "immune_myeloid_review": [
        "C1QA",
        "C1QB",
        "C1QC",
        "TYROBP",
        "LYZ",
        "CSF1R",
        "HLA-DRA",
        "CD74",
        "AIF1",
    ],
}


program_presence_rows = []


z_log = (
    logcpm_overall
    .copy()
)


z_log = z_log.sub(
    z_log.mean(
        axis=0
    ),
    axis=1,
)


z_log = z_log.div(
    z_log.std(
        axis=0
    )
    .replace(
        0,
        np.nan,
    ),
    axis=1,
)


program_scores = (
    pb_meta_f[
        [
            "sample_id",
            "condition",
            "n_cells",
            "library_size",
        ]
    ]
    .copy()
)


for program, markers in PROGRAMS.items():
    present = [
        gene
        for gene in markers
        if gene in z_log.columns
    ]

    missing = [
        gene
        for gene in markers
        if gene not in z_log.columns
    ]


    program_presence_rows.append(
        {
            "program": program,
            "n_markers_total": len(
                markers
            ),
            "n_markers_present": len(
                present
            ),
            "n_markers_missing": len(
                missing
            ),
            "markers_present": ",".join(
                present
            ),
            "markers_missing": ",".join(
                missing
            ),
        }
    )


    program_scores[
        f"{program}_score"
    ] = (
        z_log[
            present
        ]
        .mean(
            axis=1
        )
        .values
        if present
        else np.nan
    )


program_presence = pd.DataFrame(
    program_presence_rows
)


write_tsv_replace(
    program_presence,
    METADATA_DIR
    / f"{OUT_PREFIX}_program_marker_presence.tsv",
)


write_tsv_replace(
    program_scores
    .reset_index(
        names="group_key"
    ),
    METADATA_DIR
    / f"{OUT_PREFIX}_sample_level_program_scores.tsv",
)


program_test_rows = []
program_pair_rows = []


conditions = sorted(
    program_scores[
        "condition"
    ]
    .astype(str)
    .unique()
)


for program in PROGRAMS.keys():
    score_col = (
        f"{program}_score"
    )


    groups = [
        program_scores.loc[
            program_scores[
                "condition"
            ]
            .astype(str)
            == condition,
            score_col,
        ].values
        for condition
        in conditions
    ]


    p_value = safe_kruskal(
        groups
    )


    program_test_rows.append(
        {
            "program": program,
            "score_col": score_col,
            "test": "Kruskal-Wallis",
            "pvalue": p_value,
            **{
                f"median_{condition}": (
                    float(
                        np.nanmedian(
                            group
                        )
                    )
                    if len(
                        group
                    )
                    else np.nan
                )
                for condition, group
                in zip(
                    conditions,
                    groups,
                )
            },
        }
    )


    for i in range(
        len(
            conditions
        )
    ):
        for j in range(
            i + 1,
            len(
                conditions
            ),
        ):
            condition_a = conditions[
                i
            ]

            condition_b = conditions[
                j
            ]


            xa = program_scores.loc[
                program_scores[
                    "condition"
                ]
                .astype(str)
                == condition_a,
                score_col,
            ].values


            xb = program_scores.loc[
                program_scores[
                    "condition"
                ]
                .astype(str)
                == condition_b,
                score_col,
            ].values


            program_pair_rows.append(
                {
                    "program": program,
                    "condition_A": condition_a,
                    "condition_B": condition_b,
                    "n_A": int(
                        np.isfinite(
                            xa
                        ).sum()
                    ),
                    "n_B": int(
                        np.isfinite(
                            xb
                        ).sum()
                    ),
                    "median_A": (
                        float(
                            np.nanmedian(
                                xa
                            )
                        )
                        if len(
                            xa
                        )
                        else np.nan
                    ),
                    "median_B": (
                        float(
                            np.nanmedian(
                                xb
                            )
                        )
                        if len(
                            xb
                        )
                        else np.nan
                    ),
                    "delta_median_A_minus_B": (
                        float(
                            np.nanmedian(
                                xa
                            )
                            - np.nanmedian(
                                xb
                            )
                        )
                        if (
                            len(
                                xa
                            )
                            and len(
                                xb
                            )
                        )
                        else np.nan
                    ),
                    "pvalue_mannwhitney": safe_mannwhitney(
                        xa,
                        xb,
                    ),
                }
            )


program_global = pd.DataFrame(
    program_test_rows
)


program_global[
    "padj"
] = benjamini_hochberg(
    program_global[
        "pvalue"
    ].values
)


program_global = program_global.sort_values(
    [
        "padj",
        "pvalue",
    ],
    na_position="last",
)


program_pairwise = pd.DataFrame(
    program_pair_rows
)


program_pairwise[
    "padj_mannwhitney"
] = benjamini_hochberg(
    program_pairwise[
        "pvalue_mannwhitney"
    ].values
)


program_pairwise = program_pairwise.sort_values(
    [
        "padj_mannwhitney",
        "pvalue_mannwhitney",
    ],
    na_position="last",
)


write_tsv_replace(
    program_global,
    METADATA_DIR
    / f"{OUT_PREFIX}_program_score_global_kruskal.tsv",
)


write_tsv_replace(
    program_pairwise,
    METADATA_DIR
    / f"{OUT_PREFIX}_program_score_pairwise_mannwhitney.tsv",
)


EXPECTED_PROGRAM_SHAPES = {
    "presence": (
        7,
        6,
    ),
    "scores": (
        77,
        11,
    ),
    "global": (
        7,
        9,
    ),
    "pairwise": (
        42,
        10,
    ),
}


# The saved score table has one extra group_key column after reset_index.
saved_program_scores_shape = (
    program_scores.shape[
        0
    ],
    program_scores.shape[
        1
    ]
    + 1,
)


observed_program_shapes = {
    "presence": program_presence.shape,
    "scores": saved_program_scores_shape,
    "global": program_global.shape,
    "pairwise": program_pairwise.shape,
}


if observed_program_shapes != {
    "presence": (
        7,
        6,
    ),
    "scores": (
        77,
        12,
    ),
    "global": (
        7,
        9,
    ),
    "pairwise": (
        42,
        10,
    ),
}:
    raise RuntimeError(
        "Historical program-score output shape checkpoint failed. "
        f"Observed: {observed_program_shapes}"
    )


display(
    program_global
)


display(
    program_pairwise.head(
        20
    )
)


print(
    "PASS: historical program-score outputs reproduced."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_program_marker_presence.tsv shape=(7, 6)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_sample_level_program_scores.tsv shape=(77, 12)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_program_score_global_kruskal.tsv shape=(7, 9)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_program_score_pairwise_mannwhitney.tsv shape=(42, 10)


,program,score_col,test,pvalue,median_GBM,median_LGG,median_ndGBM,median_rGBM,padj
3,MES_like,MES_like_score,Kruskal-Wallis,0.006858,0.080356,-2.272040,-0.279584,0.121846,0.037022
4,hypoxia_stress,hypoxia_stress_score,Kruskal-Wallis,0.020526,0.139258,-1.257060,0.014455,0.063136,0.037022
0,OPC_proneural,OPC_proneural_score,Kruskal-Wallis,0.022535,0.022480,1.117839,0.054328,0.170847,0.037022
5,cycling,cycling_score,Kruskal-Wallis,0.025211,0.203176,-0.427764,0.318749,0.424719,0.037022
1,NPC_neural,NPC_neural_score,Kruskal-Wallis,0.026444,-0.099047,0.879344,-0.147974,-0.264109,0.037022
6,immune_myeloid_review,immune_myeloid_review_score,Kruskal-Wallis,0.036276,0.383883,-0.875764,-0.203131,-0.055433,0.042322
2,AC_reactive,AC_reactive_score,Kruskal-Wallis,0.098234,-0.042457,-1.014897,0.253113,0.142264,0.098234


,program,condition_A,condition_B,n_A,n_B,median_A,median_B,delta_median_A_minus_B,pvalue_mannwhitney,padj_mannwhitney
18,MES_like,GBM,LGG,39,4,0.080356,-2.272040,2.352396,0.000016,0.000681
21,MES_like,LGG,ndGBM,4,18,-2.272040,-0.279584,-1.992457,0.000273,0.004594
22,MES_like,LGG,rGBM,4,16,-2.272040,0.121846,-2.393886,0.000413,0.004594
24,hypoxia_stress,GBM,LGG,39,4,0.139258,-1.257060,1.396317,0.000438,0.004594
0,OPC_proneural,GBM,LGG,39,4,0.022480,1.117839,-1.095359,0.000616,0.005173
10,NPC_neural,LGG,rGBM,4,16,0.879344,-0.264109,1.143453,0.000826,0.005779
4,OPC_proneural,LGG,rGBM,4,16,1.117839,0.170847,0.946992,0.001651,0.008669
28,hypoxia_stress,LGG,rGBM,4,16,-1.257060,0.063136,-1.320196,0.001651,0.008669
27,hypoxia_stress,LGG,ndGBM,4,18,-1.257060,0.014455,-1.271515,0.001914,0.008931
3,OPC_proneural,LGG,ndGBM,4,18,1.117839,0.054328,1.063511,0.003281,0.013780


PASS: historical program-score outputs reproduced.


In [11]:
# =========================
# 10. Historical figures
# =========================
figure_rows = []


def save_fig(
    path,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()


    figure_rows.append(
        {
            "figure": path.name,
            "path": str(
                path
            ),
        }
    )


plot_df = (
    condition_summary
    .copy()
)


x = np.arange(
    len(
        plot_df
    )
)


plt.figure(
    figsize=(
        8,
        4,
    )
)


plt.bar(
    x,
    plot_df[
        "n_samples"
    ].values,
)


plt.xticks(
    x,
    plot_df[
        "condition"
    ].values,
    rotation=45,
    ha="right",
)


plt.ylabel(
    "Number of samples"
)


plt.title(
    "Pseudobulk samples per condition"
)


save_fig(
    FIGURES_DIR
    / f"{OUT_PREFIX}_condition_pseudobulk_sample_counts.png"
)


score_cols = [
    f"{program}_score"
    for program
    in PROGRAMS.keys()
]


heat = (
    program_scores
    .groupby(
        "condition"
    )[
        score_cols
    ]
    .median()
)


plt.figure(
    figsize=(
        10,
        max(
            3,
            0.4
            * len(
                heat
            ),
        ),
    )
)


plt.imshow(
    heat.values,
    aspect="auto",
)


plt.colorbar(
    label="Median z-score"
)


plt.yticks(
    np.arange(
        heat.shape[
            0
        ]
    ),
    heat.index,
)


plt.xticks(
    np.arange(
        heat.shape[
            1
        ]
    ),
    [
        column.replace(
            "_score",
            "",
        )
        for column
        in heat.columns
    ],
    rotation=45,
    ha="right",
)


plt.title(
    "Malignant-compartment program scores by condition"
)


save_fig(
    FIGURES_DIR
    / f"{OUT_PREFIX}_program_score_condition_medians.png"
)


top_genes = (
    global_kruskal
    .head(
        30
    )[
        "gene"
    ]
    .tolist()
)


if len(
    top_genes
) > 0:
    heat2 = (
        logcpm_overall[
            top_genes
        ]
        .copy()
    )


    order = (
        pb_meta_f
        .sort_values(
            [
                "condition",
                "sample_id",
            ]
        )
        .index
    )


    heat2 = heat2.loc[
        order
    ]


    heat2 = (
        heat2
        .sub(
            heat2.mean(
                axis=0
            ),
            axis=1,
        )
        .div(
            heat2.std(
                axis=0
            )
            .replace(
                0,
                np.nan,
            ),
            axis=1,
        )
    )


    heat2 = heat2.clip(
        -3,
        3,
    )


    plt.figure(
        figsize=(
            10,
            8,
        )
    )


    plt.imshow(
        heat2.T.values,
        aspect="auto",
        vmin=-3,
        vmax=3,
    )


    plt.colorbar(
        label="z(logCPM)"
    )


    plt.yticks(
        np.arange(
            len(
                top_genes
            )
        ),
        top_genes,
    )


    plt.xticks(
        []
    )


    plt.title(
        "Top condition-associated genes in malignant/cycling pseudobulk"
    )


    save_fig(
        FIGURES_DIR
        / f"{OUT_PREFIX}_top30_global_condition_DE_heatmap.png"
    )


figure_manifest = pd.DataFrame(
    figure_rows
)


write_tsv_replace(
    figure_manifest,
    METADATA_DIR
    / f"{OUT_PREFIX}_figure_manifest.tsv",
)


if figure_manifest.shape != (
    3,
    2,
):
    raise RuntimeError(
        f"Historical figure-manifest checkpoint failed: {figure_manifest.shape}"
    )


display(
    figure_manifest
)


print(
    "PASS: historical three-figure set reproduced."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_figure_manifest.tsv shape=(3, 2)


,figure,path
0,condition_malignant_pseudobulk_DE_HVG8000_r0_6...,G:\Repository\glioma-cnv-single-cell-atlas\fig...
1,condition_malignant_pseudobulk_DE_HVG8000_r0_6...,G:\Repository\glioma-cnv-single-cell-atlas\fig...
2,condition_malignant_pseudobulk_DE_HVG8000_r0_6...,G:\Repository\glioma-cnv-single-cell-atlas\fig...


PASS: historical three-figure set reproduced.


In [12]:
# =========================
# 11. Output manifest and final checkpoints
# =========================
expected_outputs = {
    "readiness": (
        METADATA_DIR
        / f"{OUT_PREFIX}_readiness.tsv"
    ),
    "state_cell_counts": (
        METADATA_DIR
        / f"{OUT_PREFIX}_state_cell_counts.tsv"
    ),
    "overall_malignant_sample_QC": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_sample_QC.tsv"
    ),
    "overall_malignant_condition_summary": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_condition_summary.tsv"
    ),
    "overall_malignant_counts_pseudobulk": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_counts_pseudobulk.tsv.gz"
    ),
    "overall_malignant_logCPM_pseudobulk": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_logCPM_pseudobulk.tsv.gz"
    ),
    "overall_malignant_global_kruskal": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_global_kruskal.tsv"
    ),
    "overall_malignant_pairwise_DE_test_summary": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_test_summary.tsv"
    ),
    "overall_malignant_pairwise_DE_all_genes": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_all_genes.tsv"
    ),
    "overall_malignant_pairwise_DE_top100_up_down": (
        METADATA_DIR
        / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_top100_up_down.tsv"
    ),
    "overall_malignant_pairwise_DE_significant": (
        METADATA_DIR
        / (
            f"{OUT_PREFIX}_overall_malignant_pairwise_DE_"
            "significant_FDR0_05_absLFC0_5.tsv"
        )
    ),
    "sample_state_pseudobulk_QC": (
        METADATA_DIR
        / f"{OUT_PREFIX}_sample_state_pseudobulk_QC.tsv"
    ),
    "state_condition_pseudobulk_summary": (
        METADATA_DIR
        / f"{OUT_PREFIX}_state_condition_pseudobulk_summary.tsv"
    ),
    "within_state_condition_DE_test_summary": (
        METADATA_DIR
        / f"{OUT_PREFIX}_within_state_condition_DE_test_summary.tsv"
    ),
    "within_state_condition_DE_all_genes": (
        METADATA_DIR
        / f"{OUT_PREFIX}_within_state_condition_DE_all_genes.tsv"
    ),
    "within_state_condition_DE_top50_up_down": (
        METADATA_DIR
        / f"{OUT_PREFIX}_within_state_condition_DE_top50_up_down.tsv"
    ),
    "program_marker_presence": (
        METADATA_DIR
        / f"{OUT_PREFIX}_program_marker_presence.tsv"
    ),
    "sample_level_program_scores": (
        METADATA_DIR
        / f"{OUT_PREFIX}_sample_level_program_scores.tsv"
    ),
    "program_score_global_kruskal": (
        METADATA_DIR
        / f"{OUT_PREFIX}_program_score_global_kruskal.tsv"
    ),
    "program_score_pairwise_mannwhitney": (
        METADATA_DIR
        / f"{OUT_PREFIX}_program_score_pairwise_mannwhitney.tsv"
    ),
    "figure_manifest": (
        METADATA_DIR
        / f"{OUT_PREFIX}_figure_manifest.tsv"
    ),
}


manifest = pd.DataFrame(
    [
        {
            "output_name": output_name,
            "path": str(
                path
            ),
            "exists": Path(
                path
            ).exists(),
            "size_MB": (
                round(
                    Path(
                        path
                    ).stat().st_size
                    / 1024
                    / 1024,
                    3,
                )
                if Path(
                    path
                ).exists()
                else np.nan
            ),
        }
        for output_name, path
        in expected_outputs.items()
    ]
)


write_tsv_replace(
    manifest,
    METADATA_DIR
    / f"{OUT_PREFIX}_output_manifest.tsv",
)


if manifest.shape != (
    21,
    4,
):
    raise RuntimeError(
        f"Historical output-manifest checkpoint failed: {manifest.shape}"
    )


if not manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more expected Notebook-22 outputs are missing."
    )


display(
    manifest
)


print(
    "\nFINAL CHECKPOINTS"
)


print(
    "Historical notebook mapping:",
    "21_condition_specific_malignant_pseudobulk_DE_HVG8000_r0_6_"
    "v3_fixed_duplicate_state_column.ipynb -> 22_condition_pseudobulk.ipynb",
)


print(
    "Canonical Notebook-20 input:",
    INPUT_H5AD,
)


print(
    "Canonical Notebook-21 pseudobulk input:",
    UPSTREAM_PSEUDOBULK_H5AD,
)


print(
    "Source shape:",
    (
        obs.shape[
            0
        ],
        var.shape[
            0
        ],
    ),
)


print(
    "Selected malignant/cycling cells:",
    int(
        malig_mask.sum()
    ),
)


print(
    "State cell counts:",
    observed_selected_state_counts,
)


print(
    "Readiness:",
    readiness.shape,
)


print(
    "Overall pseudobulk samples / passing:",
    (
        pb_counts.shape[
            0
        ],
        pb_counts_f.shape[
            0
        ],
    ),
)


print(
    "Overall genes tested:",
    len(
        genes_test
    ),
)


print(
    "Overall sample QC:",
    sample_qc.shape,
)


print(
    "Overall condition summary:",
    condition_summary.shape,
)


print(
    "Overall Kruskal-Wallis:",
    global_kruskal.shape,
)


print(
    "Overall pairwise summary:",
    pairwise_summary.shape,
)


print(
    "Overall pairwise all genes:",
    pairwise_de.shape,
)


print(
    "Overall top100 up/down:",
    top100_overall.shape,
)


print(
    "Overall significant:",
    sig_overall.shape,
)


print(
    "Sample-state pseudobulk QC:",
    state_pb_qc.shape,
)


print(
    "State-condition summary:",
    state_condition_summary.shape,
)


print(
    "Within-state DE summary:",
    within_state_summary.shape,
)


print(
    "Within-state all genes:",
    within_state_de.shape,
)


print(
    "Within-state top50 up/down:",
    within_state_top50.shape,
)


print(
    "Program marker presence:",
    program_presence.shape,
)


print(
    "Sample-level program scores:",
    saved_program_scores_shape,
)


print(
    "Program global Kruskal-Wallis:",
    program_global.shape,
)


print(
    "Program pairwise Mann-Whitney:",
    program_pairwise.shape,
)


print(
    "Figure manifest:",
    figure_manifest.shape,
)


print(
    "Output manifest:",
    manifest.shape,
)


print(
    "\n[DONE] Notebook 22 canonical condition-specific malignant pseudobulk DE completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\condition_malignant_pseudobulk_DE_HVG8000_r0_6_output_manifest.tsv shape=(21, 4)


,output_name,path,exists,size_MB
0,readiness,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
1,state_cell_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
2,overall_malignant_sample_QC,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.007
3,overall_malignant_condition_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
4,overall_malignant_counts_pseudobulk,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,1.750
5,overall_malignant_logCPM_pseudobulk,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,4.583
6,overall_malignant_global_kruskal,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,2.004
7,overall_malignant_pairwise_DE_test_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
8,overall_malignant_pairwise_DE_all_genes,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,14.801
9,overall_malignant_pairwise_DE_top100_up_down,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.214



FINAL CHECKPOINTS
Historical notebook mapping: 21_condition_specific_malignant_pseudobulk_DE_HVG8000_r0_6_v3_fixed_duplicate_state_column.ipynb -> 22_condition_pseudobulk.ipynb
Canonical Notebook-20 input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad
Canonical Notebook-21 pseudobulk input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_malignant_state_pseudobulk_logCPM.h5ad
Source shape: (351427, 15176)
Selected malignant/cycling cells: 161884
State cell counts: {'OPC_NPC_like_malignant': 61646, 'cycling_malignant_candidate': 38634, 'hypoxia_stress_malignant': 32896, 'AC_like_reactive_malignant': 22525, 'NPC_neuronal_like_malignant': 6183}
Readiness: (10, 2)
Overall pseudobulk samples / passing: (80, 77)
Overall genes tested: 15081
Overall sample QC: (77, 10)
Overall condition summary: (4, 6)
Overall Kruskal-Wallis: (15081, 9)
Overall pairwise summary: (6, 10)
Ove

## Expected historical checkpoints

A successful run should end with:

```text
Source shape: (351427, 15176)
Selected malignant/cycling cells: 161884

State cell counts:
    OPC_NPC_like_malignant          61646
    cycling_malignant_candidate     38634
    hypoxia_stress_malignant        32896
    AC_like_reactive_malignant      22525
    NPC_neuronal_like_malignant      6183

Readiness: (10, 2)

Overall pseudobulk samples / passing: (80, 77)
Overall genes tested: 15081
Overall sample QC: (77, 10)
Overall condition summary: (4, 6)
Overall Kruskal-Wallis: (15081, 9)
Overall pairwise summary: (6, 10)
Overall pairwise all genes: (90486, 13)
Overall top100 up/down: (1200, 14)
Overall significant: (27004, 13)

Sample-state pseudobulk QC: (290, 10)
State-condition summary: (19, 6)
Within-state DE summary: (27, 11)
Within-state all genes: (364224, 14)
Within-state top50 up/down: (2400, 15)

Program marker presence: (7, 6)
Sample-level program scores: (77, 12)
Program global Kruskal-Wallis: (7, 9)
Program pairwise Mann-Whitney: (42, 10)

Figure manifest: (3, 2)
Output manifest: (21, 4)
```

The notebook also hard-checks the historical overall condition summaries, all six overall pairwise DE significance totals, all 19 state-by-condition pseudobulk summaries, the 24 tested and 3 skipped within-state contrasts, and the complete canonical state-cell counts.